# From Voice to Vision — Fase 8: Speaker-Leakage Experiment & Final Artifacts

Questo notebook conclude il progetto e produce il materiale definitivo per il paper:

1. **Esperimento di speaker leakage** — riallena la *stessa* rete con uno split **casuale**
   (stesse dimensioni e stesse proporzioni di classe, ma senza separazione per attore) e mostra
   quanto l'accuracy risulti *artificialmente gonfiata*. È la prova che il nostro 0.588
   speaker-independent è un risultato onesto.
2. **Rigenerazione di tutte le figure con etichette in inglese**, per coerenza con il paper.
3. **Salvataggio degli artefatti** in `results/` (metriche JSON + figure) da versionare su GitHub.

⏱️ Tempo totale ≈ 20 minuti su GPU.

In [ ]:
# === 1. Setup ===
REPO_URL = "https://github.com/Nadaa3672/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git","")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm diffusers transformers accelerate

import numpy as np, torch, json
import matplotlib.pyplot as plt
from src import config, data_loader, features
from src.models import cnn

data_loader.download_ravdess()
df = data_loader.build_index()
data = features.build_dataset(df, denoise=False, cache=True)
data["split"] = df["split"].to_numpy()
splits = features.split_arrays(data)
device = cnn.get_device()

FIG = config.FIGURES_DIR; FIG.mkdir(parents=True, exist_ok=True)
RES = config.RESULTS_DIR; RES.mkdir(parents=True, exist_ok=True)
print("Setup OK — device:", device)

## Parte 1 — Speaker-Leakage Experiment

Costruiamo uno split **casuale** con le stesse identiche dimensioni (1080/120/240) e le stesse
proporzioni di classe di quello speaker-independent. L'unica differenza è che le clip dello
**stesso attore possono finire sia in training sia in test**. Alleniamo la stessa CNN, con gli
stessi iperparametri e lo stesso numero di epoche: qualsiasi differenza di accuracy è quindi
attribuibile *solo* al leakage dell'identità del parlante.

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(data["y"])); y = data["y"]
idx_tr, idx_tmp = train_test_split(idx, test_size=360, stratify=y, random_state=config.SEED)
idx_val, idx_te = train_test_split(idx_tmp, test_size=240, stratify=y[idx_tmp], random_state=config.SEED)

split_random = np.empty(len(idx), dtype=object)
split_random[idx_tr] = "train"; split_random[idx_val] = "val"; split_random[idx_te] = "test"
data_rand = dict(data); data_rand["split"] = split_random
splits_rand = features.split_arrays(data_rand)

print("Random split:", {s: len(splits_rand[s]['y']) for s in ['train','val','test']})
# quanti attori del test compaiono anche nel training? (dovrebbero essere tutti)
actors = df["actor"].to_numpy()
shared = len(set(actors[idx_te]) & set(actors[idx_tr]))
print(f"Attori del test presenti anche nel training: {shared}/{len(set(actors[idx_te]))}  <-- il leakage")

In [ ]:
# Stessa architettura, stessi iperparametri, stesso training del modello speaker-independent
HP = {"lr": 1e-3, "dropout": 0.3, "weight_decay": 1e-4, "batch_size": 32, "width": 32}

print("=== Training with RANDOM split (speaker leakage) ===")
leak = cnn.train_cnn(splits_rand, HP, epochs=60, patience=12, deltas=True, augment=True, verbose=False)
print(f"Random split      → test accuracy = {leak['test_acc']:.3f}")

print("\n=== Training with SPEAKER-INDEPENDENT split (our protocol) ===")
sind = cnn.train_cnn(splits, HP, epochs=60, patience=12, deltas=True, augment=True, verbose=False)
print(f"Speaker-independent → test accuracy = {sind['test_acc']:.3f}")

gap = leak['test_acc'] - sind['test_acc']
print(f"\n*** Inflazione dovuta al leakage: +{gap*100:.1f} punti percentuali ***")

In [ ]:
# Figura: confronto dei due protocolli
fig, ax = plt.subplots(figsize=(6.4,3.8))
names = ["Random split\n(speaker leakage)", "Speaker-independent\n(our protocol)"]
vals  = [leak['test_acc'], sind['test_acc']]
bars = ax.bar(names, vals, color=["#c44e52", "#2f6db0"], edgecolor="white", width=.6)
for b,v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.012, f"{v:.3f}", ha="center", fontsize=11, weight="bold")
ax.annotate("", xy=(0,vals[0]-0.01), xytext=(0,vals[1]+0.01),
            arrowprops=dict(arrowstyle="<->", color="black", lw=1.2))
ax.text(0.06, (vals[0]+vals[1])/2, f"+{gap*100:.1f} pts\ninflation", fontsize=9, va="center")
ax.set_ylabel("Test accuracy"); ax.set_ylim(0,1.0)
ax.set_title("Effect of speaker leakage on reported accuracy")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig(FIG/"08_leakage.png", dpi=150); plt.show()

json.dump({"random_split_test_acc": float(leak['test_acc']),
           "speaker_independent_test_acc": float(sind['test_acc']),
           "inflation_points": float(gap*100)},
          open(RES/"leakage_experiment.json","w"), indent=2)
print("✓ salvato results/leakage_experiment.json")

## Parte 2 — Figure in inglese

Rigeneriamo con etichette in inglese le figure che finiscono nel paper.

In [ ]:
# --- Fig. preprocessing: raw vs denoised ---
import librosa, librosa.display
from src import preprocessing

row = df[df.emotion == "happy"].iloc[0]
raw = preprocessing.load_audio(row.path)
den = preprocessing.reduce_noise(raw)

fig, ax = plt.subplots(2, 2, figsize=(13,7))
for j,(sig,ttl) in enumerate([(raw,"Raw"),(den,"Denoised")]):
    librosa.display.waveshow(sig, sr=config.SAMPLE_RATE, ax=ax[0,j])
    ax[0,j].set_title(f"Waveform — {ttl}"); ax[0,j].set_xlabel("Time (s)"); ax[0,j].set_ylabel("Amplitude")
    S = librosa.power_to_db(librosa.feature.melspectrogram(
        y=sig, sr=config.SAMPLE_RATE, n_mels=config.N_MELS,
        n_fft=config.N_FFT, hop_length=config.HOP_LENGTH), ref=np.max)
    librosa.display.specshow(S, sr=config.SAMPLE_RATE, hop_length=config.HOP_LENGTH,
                             x_axis="time", y_axis="mel", ax=ax[1,j])
    ax[1,j].set_title(f"log-Mel spectrogram — {ttl}"); ax[1,j].set_xlabel("Time (s)")
plt.tight_layout(); plt.savefig(FIG/"en_preprocessing.png", dpi=150); plt.show()

In [ ]:
# --- Fig. mean spectrogram per emotion ---
fig, axes = plt.subplots(2, 4, figsize=(16,7))
for i, emo in enumerate(config.EMOTIONS):
    ax = axes[i//4, i%4]
    mask = (data["y"] == config.EMOTION_TO_ID[emo])
    librosa.display.specshow(data["X_mel"][mask].mean(0), sr=config.SAMPLE_RATE,
                             hop_length=config.HOP_LENGTH, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(f"{emo}  (n={mask.sum()})"); ax.set_xlabel("Time (s)"); ax.label_outer()
plt.suptitle("Class-averaged log-Mel spectrograms (RAVDESS)", fontsize=14)
plt.tight_layout(); plt.savefig(FIG/"en_mean_spectrograms.png", dpi=150); plt.show()

In [ ]:
# --- Carica il modello ottimizzato (ricrea il checkpoint se assente) ---
ckpt = RES/"best_cnn.pt"
if not ckpt.exists():
    print("Checkpoint assente: uso il modello speaker-independent appena allenato.")
    torch.save({"state_dict": sind["model"].state_dict(), "hp": HP,
                "norm": sind["norm"], "in_channels": sind["in_channels"]}, ckpt)
model, ck = cnn.load_checkpoint(ckpt, device)
mean, std = ck["norm"]
loader = cnn.eval_loader(splits, "test", mean, std, deltas=True, batch_size=32)
acc, preds, tgts = cnn.evaluate(model, loader, device)
print(f"Model loaded — test acc = {acc:.3f}")

In [ ]:
# --- Fig. Grad-CAM (English labels) ---
from src.explainability.gradcam import GradCAM, last_conv_block
from src.models.cnn import MelDataset

test_ds = MelDataset(splits["test"]["X_mel"], splits["test"]["y"], mean, std, deltas=True, augment=False)
cam_engine = GradCAM(model, last_conv_block(model))

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, emo in enumerate(config.EMOTIONS):
    ax = axes[i//4, i%4]
    cls = config.EMOTION_TO_ID[emo]
    ok = np.where((tgts == cls) & (preds == tgts))[0]
    j = (ok if len(ok) else np.where(tgts == cls)[0])[0]
    x, _ = test_ds[j]; x = x.unsqueeze(0).to(device)
    cam, _ = cam_engine(x, class_idx=cls)
    ax.imshow(test_ds.X[j,0], origin="lower", aspect="auto", cmap="magma")
    ax.imshow(cam, origin="lower", aspect="auto", cmap="jet", alpha=0.45)
    ax.set_title(emo); ax.set_xlabel("Time frames"); ax.set_ylabel("Mel bands"); ax.label_outer()
plt.suptitle("Grad-CAM — time–frequency regions driving the decision", fontsize=14)
plt.tight_layout(); plt.savefig(FIG/"en_gradcam.png", dpi=150); plt.show()

In [ ]:
# --- Fig. t-SNE (English labels) ---
from src.explainability import tsne_shap
emb = tsne_shap.extract_embeddings(model, splits["test"]["X_mel"], splits["test"]["y"], mean, std)
proj = tsne_shap.run_tsne(emb)
plt.figure(figsize=(8.5,7))
for i, emo in enumerate(config.EMOTIONS):
    m = splits["test"]["y"] == i
    plt.scatter(proj[m,0], proj[m,1], label=emo, s=22, alpha=.8)
plt.legend(loc="best"); plt.title("t-SNE projection of CNN embeddings (test set)")
plt.xlabel("Dimension 1"); plt.ylabel("Dimension 2")
plt.tight_layout(); plt.savefig(FIG/"en_tsne.png", dpi=150); plt.show()

In [ ]:
# --- Fig. gallery + blend (English labels) ---
from src.art import emotion_to_image as e2i
pipe = e2i.load_pipeline()
test_df = df[df.split == "test"].reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(18, 9.5))
for i, emo in enumerate(config.EMOTIONS):
    r = test_df[test_df.emotion == emo].iloc[0]
    probs = e2i.predict_probs_from_wav(r.path, model, ck["norm"], device)
    prompt, desc = e2i.build_prompt(probs)
    img = e2i.generate(pipe, prompt, seed=config.SEED + i)
    ax = axes[i//4, i%4]; ax.imshow(img); ax.axis("off")
    ax.set_title(f"true: {emo}\npredicted: {desc}", fontsize=10)
plt.suptitle("From Voice to Vision — emotion gallery (test clips)", fontsize=15)
plt.tight_layout(); plt.savefig(FIG/"en_gallery.png", dpi=150); plt.show()

In [ ]:
# --- Fig. blended case (English labels) ---
found = None
for _, r in test_df.iterrows():
    probs = e2i.predict_probs_from_wav(r.path, model, ck["norm"], device)
    if np.sort(probs)[::-1][1] >= 0.25:
        found = (r, probs); break

if found:
    r, probs = found
    prompt, desc = e2i.build_prompt(probs)
    img = e2i.generate(pipe, prompt, seed=config.SEED)
    fig, ax = plt.subplots(1, 2, figsize=(13,5))
    ax[0].bar(config.EMOTIONS, probs, color="#4C78A8")
    ax[0].set_title(f"Predicted probabilities (true label: {r.emotion})")
    ax[0].set_ylabel("Probability"); ax[0].tick_params(axis='x', rotation=40)
    ax[1].imshow(img); ax[1].axis("off"); ax[1].set_title(f"Blended artwork: {desc}", fontsize=11)
    plt.tight_layout(); plt.savefig(FIG/"en_blend.png", dpi=150); plt.show()
else:
    print("Nessun caso ambiguo trovato sopra la soglia 0.25")

## Parte 3 — Salvataggio artefatti e download

Salviamo le metriche in `results/` e scarichiamo tutto per caricarlo su GitHub.

In [ ]:
# Riepilogo metriche del progetto (valori consolidati dagli esperimenti)
summary = {
  "protocol": "speaker-independent (16 train / 4 val / 4 test actors)",
  "dataset": "RAVDESS speech, 1440 clips, 8 emotions",
  "chance_level": 0.125,
  "classical_baselines": {"SVM_RBF": 0.517, "LogisticRegression": 0.487,
                          "RandomForest": 0.463, "KNN_k7": 0.329},
  "cnn_baseline_no_augment": 0.529,
  "finalists": {"default": {"val": 0.562, "test": 0.537},
                "PSO":     {"val": 0.512, "test": 0.567},
                "FSO":     {"val": 0.504, "test": 0.583},
                "GA":      {"val": 0.546, "test": 0.554}},
  "ensemble_test_acc": 0.588,
  "leakage_experiment": {"random_split": float(leak['test_acc']),
                         "speaker_independent": float(sind['test_acc'])}
}
json.dump(summary, open(RES/"summary_results.json","w"), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# Scarica results/ (modello + metriche + figure) da caricare su GitHub
import shutil
from google.colab import files
shutil.make_archive('/content/results_bundle', 'zip', str(RES))
files.download('/content/results_bundle.zip')

### ✅ Fatto
Carica su GitHub il contenuto di `results_bundle.zip` dentro la cartella **`results/`**
(comprese le figure in `results/figures/`), e inviami le figure `en_*.png`: le inserisco nel paper
al posto di quelle in italiano, insieme alla nuova sezione sull'esperimento di leakage.